# Hybrid RAG Pipeline
### PDF → Chunking → Dense (FAISS) + Sparse (BM25) → Fusion → Answer

In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf rank_bm25 -q

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from rank_bm25 import BM25Okapi
import numpy as np

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [2]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")
print(f"Sample Chunk:\n{chunks[0].page_content}")

Total Chunks: 136
Sample Chunk:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director General of Defence Research and
Development Organisation
In office
1992–1999
A. P. J. Abdul Kalam
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl
kəˈlɑːm/ ⓘ UB-duul k ə -LAHM; 15 October 1931 –
27 July 2015) was an Indian aerospace scientist and
statesman who served as the president of India from
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a


## Step 3: Create Dense Search (FAISS)

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Create Sparse Search (BM25)

In [5]:
tokenized_chunks = [doc.page_content.lower().split() for doc in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 Index Created with {len(tokenized_chunks)} documents")

BM25 Index Created with 136 documents


## Step 5: Hybrid Retrieval with Alpha-Weighted Fusion

In [6]:
def hybrid_search(query, chunks, vector_store, bm25, k=3, alpha=0.5):
    """
    alpha = 1.0 → pure dense search
    alpha = 0.0 → pure sparse search
    """
    # Dense search: get scores for all chunks
    query_embedding = embeddings.embed_query(query)
    dense_results = vector_store.similarity_search_with_score(query, k=len(chunks))
    
    # Build dense score map (convert distance to similarity)
    dense_scores = {}
    for doc, score in dense_results:
        dense_scores[doc.page_content] = 1 / (1 + score)  # normalize distance to 0-1

    # Sparse search: BM25 scores for all chunks
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    
    # Normalize BM25 scores to 0-1
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    bm25_scores = bm25_scores / max_bm25

    # Combine scores with alpha weighting
    hybrid_scores = []
    for i, doc in enumerate(chunks):
        dense_score = dense_scores.get(doc.page_content, 0)
        sparse_score = bm25_scores[i]
        combined = alpha * dense_score + (1 - alpha) * sparse_score
        hybrid_scores.append((doc, combined, dense_score, sparse_score))

    # Sort by combined score and return top-k
    hybrid_scores.sort(key=lambda x: x[1], reverse=True)
    return hybrid_scores[:k]

## Step 6: Setup LLM & Query

In [16]:
llm = ChatOllama(model="llama3.2:3b", temperature=0.7)

query = "which color is most liked by kalam?"

# Hybrid retrieval (alpha=0.7 gives more weight to dense/semantic search)
results = hybrid_search(query, chunks, vector_store, bm25, k=3, alpha=0.7)

# Build context from retrieved chunks
context = "\n\n".join([doc.page_content for doc, _, _, _ in results])

# Create prompt
prompt = f"""Answer the question only based on the following context:

{context}.

Question: {query}
Answer:"""

# Get streaming response
print("Answer: ", end="")
for chunk in llm.stream(prompt):
    print(chunk.content, end="", flush=True)

Answer: There is no mention of Dr. A. P. J. Abdul Kalam's favorite color in the provided context.

Answer: Venkataraman was instrumental in getting the cabinet approval for allocating ₹3.88 billion (equivalent to ₹66 billion or US$780 million in 2023) for the Integrated Guided Missile Development Programme (IGMDP) and appointed Kalam as its chief executive.

## Retrieved Source Chunks (with Scores)

In [8]:
for i, (doc, combined, dense, sparse) in enumerate(results):
    print(f"\n--- Chunk {i+1} (Page {doc.metadata['page'] + 1}) ---")
    print(f"Dense: {dense:.4f} | Sparse: {sparse:.4f} | Combined: {combined:.4f}")
    print(doc.page_content[:300])


--- Chunk 1 (Page 1) ---
Dense: 0.5503 | Sparse: 1.0000 | Combined: 0.7751
Party and the then-opposition Indian National
Congress. He was widely referred to as the "People's
President". He engaged in teaching, writing and public
service after his presidency. He was a recipient of
several awards, including the Bharat Ratna, India's
highest civilian honour.
While delivering 

--- Chunk 2 (Page 11) ---
Dense: 0.5358 | Sparse: 0.8141 | Combined: 0.6749
Wheeler Island was renamed as Abdul Kalam Island.[164] In
October 2015, a 6,180 m (20,280 ft) peak near the Bara Shigri
Glacier in the Himalayas was named as Mount Kalam.[165] Dr
APJ Abdul Kalam Missile Complex, a missile research
facility in Hyderabad is named after him.[166] Dr. A. P. J.
Abdul Kal

--- Chunk 3 (Page 3) ---
Dense: 0.4874 | Sparse: 0.8400 | Combined: 0.6637
career, he was involved in the design of small
hovercraft, and remained unconvinced by his choice
of a job at DRDO.[20] Later, he joined the Indian
National Committee fo

Test Questions

1. What role did "Venkataraman" play in the missile programme funding?
2. Find all references to "Pokhran" and explain Kalam's involvement in nuclear testing.
3. What is the significance of the acronym "IGMDP" in Kalam's career?
4. Which specific "Ramanna" invited Kalam to witness the first nuclear test?
5. What was the controversy around "K. Santhanam" and the thermonuclear bomb?